In [10]:
from pathlib import Path
import os, sys, subprocess, json, time, zipfile, signal, getpass, gc
from datetime import datetime

# =========================================================
# USER SETTINGS
# =========================================================
MODE = "train"                     # "train" or "status"
RESUME_IF_AVAILABLE = True
KILL_OWN_GPU_PYTHON_PROCS = False  # set True only if you want to kill YOUR OWN stale GPU python/jupyter jobs

# Short sanity run first
KIMG = 50
TICK = 1
SNAP = 5
METRICS = "none"
SEED = 42

# Keep compatible with the 256x256 FFHQ-U StyleGAN3-R pretrained family
CFG = "stylegan3-r"
GAMMA = 1.0
CBASE = 16384
GLR = 0.0025
DLR = 0.0025
AUG = "noaug"

# Memory-safe settings
BATCH = 2
BATCH_GPU = 1
WORKERS = 1          # IMPORTANT: must be >= 1
MBSTD_GROUP = 1
MIRROR = 0

MIN_FREE_GIB_TO_START = 12.0

PRETRAINED_URL = (
    "https://api.ngc.nvidia.com/v2/models/nvidia/research/stylegan3/versions/1/"
    "files/stylegan3-r-ffhqu-256x256.pkl"
)

# =========================================================
# PATHS
# =========================================================
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()

stylegan3_repo = ROOT / "third_party" / "stylegan3"
train_py = stylegan3_repo / "train.py"
dataset_zip = ROOT / "data" / "stylegan3_zip" / "celeba_256x256.zip"

run_root = ROOT / "checkpoints" / "baseline_stylegan3r"
run_root.mkdir(parents=True, exist_ok=True)

logs_dir = ROOT / "logs" / "baseline"
logs_dir.mkdir(parents=True, exist_ok=True)

cache_dir = ROOT / ".cache"
(cache_dir / "torch_extensions").mkdir(parents=True, exist_ok=True)
(cache_dir / "dnnlib").mkdir(parents=True, exist_ok=True)

# =========================================================
# CUDA / CACHE ENV
# =========================================================
os.environ["TORCH_EXTENSIONS_DIR"] = str(cache_dir / "torch_extensions")
os.environ["DNNLIB_CACHE_DIR"] = str(cache_dir / "dnnlib")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:64"
os.environ["NCCL_P2P_DISABLE"] = "1"
os.environ["NCCL_IB_DISABLE"] = "1"

# =========================================================
# HELPERS
# =========================================================
def run_cmd(cmd, cwd=None, capture=False):
    if capture:
        return subprocess.check_output(cmd, cwd=cwd, text=True)
    print("\n[RUN]", " ".join(str(x) for x in cmd))
    subprocess.run(cmd, cwd=cwd, check=True)

def verify_dataset_zip():
    assert dataset_zip.exists(), f"Missing dataset ZIP: {dataset_zip}"
    with zipfile.ZipFile(dataset_zip, "r") as zf:
        names = zf.namelist()
        pngs = [n for n in names if n.lower().endswith(".png")]
        assert "dataset.json" in names, "dataset.json missing inside ZIP"
        assert len(pngs) > 0, "No PNG images in ZIP"
        return len(pngs)

def verify_ref_patch():
    files = {
        "bias_act": stylegan3_repo / "torch_utils" / "ops" / "bias_act.py",
        "upfirdn2d": stylegan3_repo / "torch_utils" / "ops" / "upfirdn2d.py",
        "filtered_lrelu": stylegan3_repo / "torch_utils" / "ops" / "filtered_lrelu.py",
    }
    checks = {}
    for k, p in files.items():
        text = p.read_text(encoding="utf-8", errors="ignore")
        checks[k] = "impl='ref'" in text
    return checks

def latest_run_dir(root: Path):
    dirs = [p for p in root.iterdir() if p.is_dir()] if root.exists() else []
    dirs = sorted(dirs, key=lambda p: p.stat().st_mtime, reverse=True)
    return dirs[0] if dirs else None

def latest_snapshot(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    snaps = sorted(run_dir.glob("network-snapshot-*.pkl"))
    return snaps[-1] if snaps else None

def latest_fakes(run_dir: Path):
    if run_dir is None or not run_dir.exists():
        return None
    imgs = sorted(run_dir.glob("fakes*.png"))
    return imgs[-1] if imgs else None

def tail_jsonl(path: Path, n=5):
    if not path.exists():
        return []
    lines = path.read_text(encoding="utf-8", errors="ignore").strip().splitlines()
    lines = lines[-n:]
    out = []
    for line in lines:
        try:
            out.append(json.loads(line))
        except Exception:
            out.append({"raw": line})
    return out

def get_gpu_info_torch():
    import torch
    cuda = torch.cuda.is_available()
    count = torch.cuda.device_count() if cuda else 0
    names = [torch.cuda.get_device_name(i) for i in range(count)] if cuda else []
    return cuda, count, names

def query_nvidia_smi():
    gpu_mem = []
    proc_mem = []

    out = run_cmd(
        ["nvidia-smi", "--query-gpu=index,name,memory.total,memory.used,memory.free", "--format=csv,noheader,nounits"],
        capture=True
    )
    for line in out.strip().splitlines():
        idx, name, total, used, free = [x.strip() for x in line.split(",", 4)]
        gpu_mem.append({
            "index": int(idx),
            "name": name,
            "memory_total_mib": int(total),
            "memory_used_mib": int(used),
            "memory_free_mib": int(free),
        })

    try:
        out = run_cmd(
            ["nvidia-smi", "--query-compute-apps=pid,process_name,used_memory", "--format=csv,noheader,nounits"],
            capture=True
        )
        for line in out.strip().splitlines():
            parts = [x.strip() for x in line.split(",", 2)]
            if len(parts) == 3:
                pid, pname, used = parts
                proc_mem.append({
                    "pid": int(pid),
                    "process_name": pname,
                    "used_memory_mib": int(used),
                })
    except Exception:
        pass

    return gpu_mem, proc_mem

def enrich_processes(proc_rows):
    current_user = getpass.getuser()
    enriched = []
    for row in proc_rows:
        pid = row["pid"]
        owner = "unknown"
        command = "unknown"
        try:
            owner = run_cmd(["ps", "-o", "user=", "-p", str(pid)], capture=True).strip()
        except Exception:
            pass
        try:
            command = run_cmd(["ps", "-o", "command=", "-p", str(pid)], capture=True).strip()
        except Exception:
            pass
        row2 = dict(row)
        row2["owner"] = owner
        row2["command"] = command
        row2["is_current_user"] = (owner == current_user)
        enriched.append(row2)
    return enriched

def kill_own_gpu_python_procs(proc_rows):
    current_user = getpass.getuser()
    current_pid = os.getpid()
    killed = []

    for row in proc_rows:
        pid = row["pid"]
        owner = row.get("owner", "")
        cmd = row.get("command", "").lower()

        if pid == current_pid:
            continue
        if owner != current_user:
            continue
        if ("python" not in cmd) and ("jupyter" not in cmd) and ("ipykernel" not in cmd):
            continue

        try:
            os.kill(pid, signal.SIGTERM)
            killed.append({"pid": pid, "action": "SIGTERM", "command": row.get("command", "")})
        except Exception as e:
            killed.append({"pid": pid, "action": f"SIGTERM_FAILED: {e}", "command": row.get("command", "")})

    if killed:
        time.sleep(5)
        for item in killed:
            pid = item["pid"]
            try:
                os.kill(pid, 0)
                try:
                    os.kill(pid, signal.SIGKILL)
                    item["action"] += " -> SIGKILL"
                except Exception as e:
                    item["action"] += f" -> SIGKILL_FAILED: {e}"
            except Exception:
                pass

    return killed

def build_train_cmd(resume_source):
    return [
        sys.executable, str(train_py),
        "--outdir", str(run_root),
        "--cfg", CFG,
        "--data", str(dataset_zip),
        "--gpus", "1",
        "--batch", str(BATCH),
        "--batch-gpu", str(BATCH_GPU),
        "--gamma", str(GAMMA),
        "--mirror", str(MIRROR),
        "--kimg", str(KIMG),
        "--tick", str(TICK),
        "--snap", str(SNAP),
        "--metrics", str(METRICS),
        "--seed", str(SEED),
        "--resume", str(resume_source),
        "--aug", str(AUG),
        "--cbase", str(CBASE),
        "--glr", str(GLR),
        "--dlr", str(DLR),
        "--workers", str(WORKERS),
        "--mbstd-group", str(MBSTD_GROUP),
    ]

def stream_process(cmd, cwd=None, logfile=None):
    print("\n[RUN]")
    print(" ".join(str(x) for x in cmd))
    with subprocess.Popen(
        cmd,
        cwd=cwd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        universal_newlines=True,
        env=os.environ.copy(),
    ) as proc:
        with open(logfile, "a", encoding="utf-8") if logfile else open(os.devnull, "w") as logf:
            for line in proc.stdout:
                print(line, end="")
                if logfile:
                    logf.write(line)
            proc.wait()
            if proc.returncode != 0:
                raise subprocess.CalledProcessError(proc.returncode, cmd)

# =========================================================
# PRECHECKS
# =========================================================
assert stylegan3_repo.exists(), f"Missing stylegan3 repo: {stylegan3_repo}"
assert train_py.exists(), f"Missing train.py: {train_py}"

num_zip_images = verify_dataset_zip()
patch_status = verify_ref_patch()

cuda_ok, gpu_count, gpu_names = get_gpu_info_torch()
assert cuda_ok and gpu_count > 0, "No CUDA GPU detected."

gpu_mem, proc_mem = query_nvidia_smi()
proc_mem = enrich_processes(proc_mem)

print("ROOT =", ROOT)
print("Time =", datetime.now().isoformat())
print("CUDA GPUs =", gpu_names)
print("Dataset ZIP =", dataset_zip)
print("ZIP image count =", num_zip_images)
print("Ref-patch status =", json.dumps(patch_status, indent=2))
print("CUDA env =", json.dumps({
    "PYTORCH_CUDA_ALLOC_CONF": os.environ["PYTORCH_CUDA_ALLOC_CONF"],
    "NCCL_P2P_DISABLE": os.environ["NCCL_P2P_DISABLE"],
    "NCCL_IB_DISABLE": os.environ["NCCL_IB_DISABLE"],
}, indent=2))
print("GPU memory =", json.dumps(gpu_mem, indent=2))
print("GPU compute processes =", json.dumps(proc_mem, indent=2))

if not all(patch_status.values()):
    raise RuntimeError("StyleGAN3 ref-impl patch is missing. Re-run Notebook 04 before training.")

# =========================================================
# STATUS MODE
# =========================================================
if MODE.strip().lower() == "status":
    run_dir = latest_run_dir(run_root)
    print("\nLatest run dir:", run_dir)
    if run_dir is None:
        print("No run directory found yet.")
    else:
        snap = latest_snapshot(run_dir)
        fake = latest_fakes(run_dir)
        stats_file = run_dir / "training_stats.jsonl"
        print("Latest snapshot:", snap)
        print("Latest fakes grid:", fake)
        print("Stats file exists:", stats_file.exists())
        if stats_file.exists():
            print("\nLast few stat entries:")
            for row in tail_jsonl(stats_file, n=5):
                print(json.dumps(row, indent=2))
    print("\nNotebook 05 status complete.")

# =========================================================
# TRAIN MODE
# =========================================================
elif MODE.strip().lower() == "train":
    if KILL_OWN_GPU_PYTHON_PROCS:
        killed = kill_own_gpu_python_procs(proc_mem)
        print("\nKilled own GPU python processes:", json.dumps(killed, indent=2))
        time.sleep(3)
        gc.collect()
        try:
            import torch
            torch.cuda.empty_cache()
        except Exception:
            pass
        gpu_mem, proc_mem = query_nvidia_smi()
        proc_mem = enrich_processes(proc_mem)
        print("\nGPU memory after cleanup =", json.dumps(gpu_mem, indent=2))
        print("GPU compute processes after cleanup =", json.dumps(proc_mem, indent=2))

    free_mib = gpu_mem[0]["memory_free_mib"]
    if free_mib < MIN_FREE_GIB_TO_START * 1024:
        raise RuntimeError(
            f"Not enough free GPU memory to start safely. Free memory = {free_mib/1024:.2f} GiB, "
            f"required >= {MIN_FREE_GIB_TO_START:.2f} GiB.\n"
            f"Stop other GPU jobs first, or set KILL_OWN_GPU_PYTHON_PROCS=True if those jobs are yours."
        )

    latest_run = latest_run_dir(run_root)
    latest_snap = latest_snapshot(latest_run) if latest_run is not None else None

    if RESUME_IF_AVAILABLE and latest_snap is not None:
        resume_source = latest_snap
        resume_mode = "resume_from_local_snapshot"
    else:
        resume_source = PRETRAINED_URL
        resume_mode = "resume_from_pretrained_pickle"

    cmd = build_train_cmd(resume_source)

    launch_info = {
        "time": datetime.now().isoformat(),
        "mode": resume_mode,
        "resume_source": str(resume_source),
        "dataset_zip": str(dataset_zip),
        "cfg": CFG,
        "batch": BATCH,
        "batch_gpu": BATCH_GPU,
        "gamma": GAMMA,
        "cbase": CBASE,
        "glr": GLR,
        "dlr": DLR,
        "aug": AUG,
        "workers": WORKERS,
        "mbstd_group": MBSTD_GROUP,
        "mirror": MIRROR,
        "kimg": KIMG,
        "tick": TICK,
        "snap": SNAP,
        "metrics": METRICS,
        "seed": SEED,
        "free_gpu_mib_before_launch": free_mib,
        "command": " ".join(str(x) for x in cmd),
    }

    launch_json = logs_dir / "baseline_launcher_config.json"
    launch_json.write_text(json.dumps(launch_info, indent=2), encoding="utf-8")
    print("\nSaved launcher config to:", launch_json)
    print(json.dumps(launch_info, indent=2))

    live_log = logs_dir / "baseline_train_live.log"
    stream_process(cmd, cwd=stylegan3_repo, logfile=live_log)

    latest_run = latest_run_dir(run_root)
    latest_snap = latest_snapshot(latest_run) if latest_run is not None else None
    latest_fake = latest_fakes(latest_run) if latest_run is not None else None

    print("\nTraining finished.")
    print("Latest run dir:", latest_run)
    print("Latest snapshot:", latest_snap)
    print("Latest fake grid:", latest_fake)
    print("Live log:", live_log)

else:
    raise ValueError("MODE must be 'train' or 'status'")

ROOT = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation
Time = 2026-04-07T16:37:49.433839
CUDA GPUs = ['NVIDIA H100 NVL']
Dataset ZIP = /data/Sajjan_Singh/gen/gen_scaffold/realistic_face_generation/data/stylegan3_zip/celeba_256x256.zip
ZIP image count = 202599
Ref-patch status = {
  "bias_act": true,
  "upfirdn2d": true,
  "filtered_lrelu": true
}
CUDA env = {
  "PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True,max_split_size_mb:64",
  "NCCL_P2P_DISABLE": "1",
  "NCCL_IB_DISABLE": "1"
}
GPU memory = [
  {
    "index": 0,
    "name": "NVIDIA H100 NVL",
    "memory_total_mib": 95830,
    "memory_used_mib": 28421,
    "memory_free_mib": 66815
  }
]
GPU compute processes = [
  {
    "pid": 911055,
    "process_name": "python",
    "used_memory_mib": 28368,
    "owner": "ug",
    "command": "python /data/UG/Ganesh/MS_Thesis_2/repo_2_(nix)/spectral_extrapolation_with_train_strategy/scripts/gap_fill_train.py --model iTransformer --output_dir ./OUTPUT_GAP_FILL/outputs_patch_16_